In [1]:
import sys
print(sys.executable)

import tensorflow as tf
print(tf.__version__)

f:\uni\semi 5\pattern recognition\assign_03\EN3150-CNN-Classification\.venv\Scripts\python.exe
2.21.0


In [2]:
import os
import time
import json
import random
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.21.0
Num GPUs Available: 0


In [5]:
DATA_PATH = Path("../data/processed/fashion_mnist_processed.npz")
OUTPUT_DIR = Path("outputs/model_b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 25                  # at least 20 epochs, as required
NUM_CLASSES = 10
MAX_TRAINABLE_PARAMS = 100_000

# Optimizer comparison: chosen optimizer, plain SGD, and SGD with momentum.
LEARNING_RATE = 1e-3
SGD_LEARNING_RATE = 1e-2
MOMENTUM = 0.9

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [6]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {DATA_PATH.resolve()}. "
        "Run your preprocessing notebook first, then update DATA_PATH to the saved NPZ file."
    )

archive = np.load(DATA_PATH, allow_pickle=False)
print("Available NPZ keys:", archive.files)

KEY_CANDIDATES = {
    "x_train": ["x_train", "X_train", "train_images", "X_train_processed", "x_train_processed"],
    "y_train": ["y_train", "Y_train", "train_labels", "y_train_processed"],
    "x_val":   ["x_val", "X_val", "val_images", "validation_images", "X_val_processed", "x_val_processed"],
    "y_val":   ["y_val", "Y_val", "val_labels", "validation_labels", "y_val_processed"],
    "x_test":  ["x_test", "X_test", "test_images", "X_test_processed", "x_test_processed"],
    "y_test":  ["y_test", "Y_test", "test_labels", "y_test_processed"],
}

def find_key(candidates, available):
    for key in candidates:
        if key in available:
            return key
    return None

resolved = {name: find_key(candidates, archive.files)
            for name, candidates in KEY_CANDIDATES.items()}
missing = [name for name, key in resolved.items() if key is None]
if missing:
    raise KeyError(
        f"Could not find arrays for {missing}. Available keys: {archive.files}. "
        "Update KEY_CANDIDATES to match your saved dataset."
    )

x_train = archive[resolved["x_train"]]
y_train = archive[resolved["y_train"]]
x_val   = archive[resolved["x_val"]]
y_val   = archive[resolved["y_val"]]
x_test  = archive[resolved["x_test"]]
y_test  = archive[resolved["y_test"]]

print("Resolved keys:", resolved)
print("Raw shapes:", x_train.shape, y_train.shape, x_val.shape, y_val.shape, x_test.shape, y_test.shape)

FileNotFoundError: Processed dataset not found at F:\uni\semi 5\pattern recognition\assign_03\EN3150-CNN-Classification\data\processed\fashion_mnist_processed.npz. Run your preprocessing notebook first, then update DATA_PATH to the saved NPZ file.